In [ ]:
import numpy as np
from scipy.ndimage import binary_dilation, binary_fill_holes, gaussian_filter, binary_erosion, distance_transform_edt, binary_closing, generate_binary_structure
import scipy.ndimage as ndi
from scipy.interpolate import griddata
from skimage.measure import label
from skimage.morphology import ball
from scipy.ndimage import median_filter
from collections import deque
from scipy.spatial import cKDTree
from numba import jit, prange
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor
import warnings
from skimage.measure import euler_number

# ============================================================================
# NUMBA-OPTIMIZED RASTERIZATION (10-50x faster)
# ============================================================================

@jit(nopython=True, fastmath=True)
def rasterize_triangle_numba(p1, p2, p3, volume):
    """Numba-optimized triangle rasterization. Expected speedup: 10-50x."""
    min_z = max(0, int(np.floor(min(p1[0], p2[0], p3[0]))))
    max_z = min(volume.shape[0] - 1, int(np.ceil(max(p1[0], p2[0], p3[0]))))
    min_y = max(0, int(np.floor(min(p1[1], p2[1], p3[1]))))
    max_y = min(volume.shape[1] - 1, int(np.ceil(max(p1[1], p2[1], p3[1]))))
    min_x = max(0, int(np.floor(min(p1[2], p2[2], p3[2]))))
    max_x = min(volume.shape[2] - 1, int(np.ceil(max(p1[2], p2[2], p3[2]))))

    v0 = p2 - p1
    v1 = p3 - p1
    d00 = np.dot(v0, v0)
    d01 = np.dot(v0, v1)
    d11 = np.dot(v1, v1)
    denom = d00 * d11 - d01 * d01

    if abs(denom) < 1e-10:
        return

    inv_denom = 1.0 / denom

    for z in range(min_z, max_z + 1):
        for y in range(min_y, max_y + 1):
            for x in range(min_x, max_x + 1):
                v2_0 = z - p1[0]
                v2_1 = y - p1[1]
                v2_2 = x - p1[2]
                d20 = v2_0 * v0[0] + v2_1 * v0[1] + v2_2 * v0[2]
                d21 = v2_0 * v1[0] + v2_1 * v1[1] + v2_2 * v1[2]
                v = (d11 * d20 - d01 * d21) * inv_denom
                w = (d00 * d21 - d01 * d20) * inv_denom
                u = 1.0 - v - w
                if u >= -0.01 and v >= -0.01 and w >= -0.01:
                    volume[z, y, x] = True


@jit(nopython=True, fastmath=True, parallel=True)
def rasterize_surface_numba(grid_points, volume, samples_per_edge=5):
    """
    Numba-optimized surface rasterization with parallel processing.
    Expected speedup: 20-100x faster than pure Python.
    """
    grid_resolution = grid_points.shape[0]

    for i in prange(grid_resolution - 1):
        for j in range(grid_resolution - 1):
            p1 = grid_points[i, j]
            p2 = grid_points[i+1, j]
            p3 = grid_points[i, j+1]
            p4 = grid_points[i+1, j+1]

            if (np.isnan(p1[0]) or np.isnan(p2[0]) or
                np.isnan(p3[0]) or np.isnan(p4[0])):
                continue

            for u_idx in range(samples_per_edge):
                u = u_idx / (samples_per_edge - 1) if samples_per_edge > 1 else 0.5
                for v_idx in range(samples_per_edge):
                    v = v_idx / (samples_per_edge - 1) if samples_per_edge > 1 else 0.5

                    point_0 = ((1-u)*(1-v)*p1[0] + u*(1-v)*p2[0] +
                              (1-u)*v*p3[0] + u*v*p4[0])
                    point_1 = ((1-u)*(1-v)*p1[1] + u*(1-v)*p2[1] +
                              (1-u)*v*p3[1] + u*v*p4[1])
                    point_2 = ((1-u)*(1-v)*p1[2] + u*(1-v)*p2[2] +
                              (1-u)*v*p3[2] + u*v*p4[2])

                    iz = int(np.round(point_0))
                    iy = int(np.round(point_1))
                    ix = int(np.round(point_2))

                    if (0 <= iz < volume.shape[0] and
                        0 <= iy < volume.shape[1] and
                        0 <= ix < volume.shape[2]):
                        volume[iz, iy, ix] = True


@jit(nopython=True, fastmath=True)
def check_triangle_in_bounds(p1, p2, p3, shape):
    """Check if triangle intersects volume bounds."""
    min_z = min(p1[0], p2[0], p3[0])
    max_z = max(p1[0], p2[0], p3[0])
    min_y = min(p1[1], p2[1], p3[1])
    max_y = max(p1[1], p2[1], p3[1])
    min_x = min(p1[2], p2[2], p3[2])
    max_x = max(p1[2], p2[2], p3[2])

    if max_z < 0 or min_z >= shape[0]: return False
    if max_y < 0 or min_y >= shape[1]: return False
    if max_x < 0 or min_x >= shape[2]: return False
    return True


# ============================================================================
# VECTORIZED OVERLAP DETECTION (5-10x faster)
# ============================================================================

def detect_overlaps_vectorized(fitted_sheets, num_components):
    """
    Vectorized overlap detection using scipy operations.
    Expected speedup: 5-10x faster than sequential method.
    """
    shape = list(fitted_sheets.values())[0].shape
    count_map = np.zeros(shape, dtype=np.int32)
    for i in range(1, num_components + 1):
        count_map += fitted_sheets[i].astype(np.int32)

    potential_overlap = count_map > 1
    if not np.any(potential_overlap):
        return np.zeros(shape, dtype=bool)

    labeled_result = np.zeros(shape, dtype=np.int32)
    for i in range(1, num_components + 1):
        labeled_result[fitted_sheets[i]] = i

    from scipy.ndimage import generic_filter

    def has_different_neighbor(values):
        center = values[13]
        if center == 0:
            return 0
        for val in values:
            if val > 0 and val != center:
                return 1
        return 0

    overlap_mask = np.zeros(shape, dtype=bool)
    coords = np.column_stack(np.nonzero(potential_overlap))
    if len(coords) == 0:
        return overlap_mask

    min_coords = np.maximum(coords.min(axis=0) - 1, 0)
    max_coords = np.minimum(coords.max(axis=0) + 2, shape)
    slices = tuple(slice(min_coords[i], max_coords[i]) for i in range(3))
    roi_labeled = labeled_result[slices]
    roi_potential = potential_overlap[slices]

    roi_overlap = generic_filter(
        roi_labeled,
        has_different_neighbor,
        size=3,
        mode='constant',
        cval=0
    ).astype(bool)

    roi_overlap = roi_overlap & roi_potential
    overlap_mask[slices] = roi_overlap
    return overlap_mask


# ============================================================================
# ALGORITHMIC OPTIMIZATIONS
# ============================================================================

def adaptive_grid_resolution(component, base_resolution=100, max_resolution=150):
    """Dynamically adjust grid resolution based on component size."""
    num_voxels = np.sum(component)
    if num_voxels < 500:
        return min(30, base_resolution)
    elif num_voxels < 2000:
        return min(50, base_resolution)
    elif num_voxels < 5000:
        return min(70, base_resolution)
    elif num_voxels < 15000:
        return base_resolution
    else:
        return min(max_resolution, base_resolution + 20)


def should_skip_smoothing(component, coverage_threshold=0.8):
    """Determine if a component needs smoothing based on planarity."""
    coords = np.column_stack(np.nonzero(component))
    coords_mean = coords.mean(axis=0)
    U, S, Vt = np.linalg.svd(coords - coords_mean, full_matrices=False)
    if S[2] / S[0] < 0.05:
        return True
    return False


def zero_volume_faces(volume, thickness=5):
    """Optimized face zeroing using slicing."""
    result = volume.copy()
    result[:thickness, :, :] = False
    result[-thickness:, :, :] = False
    result[:, :thickness, :] = False
    result[:, -thickness:, :] = False
    result[:, :, :thickness] = False
    result[:, :, -thickness:] = False
    return result


# ============================================================================
# OPTIMIZED MAIN FITTING FUNCTION
# ============================================================================

def fit_curved_sheet_to_component_optimized(
    component,
    grid_resolution=100,
    thickness=3,
    smoothing=1.0,
    use_median_filter=True,
    max_distance=10,
    use_numba=True,
    adaptive_resolution=True,
    samples_per_edge=8
):
    """
    OPTIMIZED version of fit_curved_sheet_to_component.
    Key optimizations:
    1. Numba JIT compilation for rasterization (10-50x speedup)
    2. Adaptive grid resolution (2-4x speedup for small components)
    3. Skip smoothing when not needed
    """
    coords = np.column_stack(np.nonzero(component))
    if len(coords) < 10:
        return component.copy()

    if adaptive_resolution:
        grid_resolution = adaptive_grid_resolution(component, grid_resolution)
        print(f"    Using adaptive grid resolution: {grid_resolution}")

    coords_mean = coords.mean(axis=0)
    U, S, Vt = np.linalg.svd(coords - coords_mean, full_matrices=False)
    tangent1, tangent2 = Vt[0], Vt[1]
    normal_guess = Vt[2]

    uv_coords = (coords - coords_mean) @ np.column_stack([tangent1, tangent2])
    w_coords = (coords - coords_mean) @ normal_guess

    if len(coords) > 5000:
        indices = np.random.choice(len(coords), 5000, replace=False)
        uv_coords_sample = uv_coords[indices]
        w_coords_sample = w_coords[indices]
    else:
        uv_coords_sample = uv_coords
        w_coords_sample = w_coords

    u_min, u_max = uv_coords[:,0].min(), uv_coords[:,0].max()
    v_min, v_max = uv_coords[:,1].min(), uv_coords[:,1].max()
    u_padding = (u_max - u_min) * 0.05
    v_padding = (v_max - v_min) * 0.05

    grid_u, grid_v = np.meshgrid(
        np.linspace(u_min - u_padding, u_max + u_padding, num=grid_resolution),
        np.linspace(v_min - v_padding, v_max + v_padding, num=grid_resolution),
        indexing='ij'
    )

    try:
        w_grid = griddata(uv_coords_sample, w_coords_sample, (grid_u, grid_v), method='linear')
    except:
        w_grid = griddata(uv_coords_sample, w_coords_sample, (grid_u, grid_v), method='nearest')

    if np.any(np.isnan(w_grid)):
        mask = np.isnan(w_grid)
        w_grid_nearest = griddata(uv_coords_sample, w_coords_sample, (grid_u, grid_v), method='nearest')
        w_grid[mask] = w_grid_nearest[mask]

    if use_median_filter:
        w_grid = median_filter(w_grid, size=3)

    skip_smooth = should_skip_smoothing(component)
    
    if smoothing > 0 and not skip_smooth:
        w_grid = gaussian_filter(w_grid, sigma=smoothing)
    elif skip_smooth:
        print(f"    Skipping smoothing (component already planar)")

    # grid_padding = 0.02
    # if grid_padding is not None:
    #     u_data_min, u_data_max = uv_coords[:,0].min(), uv_coords[:,0].max()
    #     v_data_min, v_data_max = uv_coords[:,1].min(), uv_coords[:,1].max()
    #     u_range = u_data_max - u_data_min
    #     v_range = v_data_max - v_data_min
    #     u_pad = u_range * grid_padding
    #     v_pad = v_range * grid_padding
    #     grid_mask = ((grid_u >= u_data_min - u_pad) & (grid_u <= u_data_max + u_pad) &
    #                  (grid_v >= v_data_min - v_pad) & (grid_v <= v_data_max + v_pad))
    #     w_grid[~grid_mask] = np.nan

    #Grid trimming with KDTree (already optimized in original)
    tree = cKDTree(uv_coords)
    threshold = (u_max - u_min + v_max - v_min) / (2 * grid_resolution) * 2
    
    grid_uv_flat = np.column_stack([grid_u.ravel(), grid_v.ravel()])
    distances, _ = tree.query(grid_uv_flat, k=1)
    distances = distances.reshape(grid_resolution, grid_resolution)
    
    original_data_mask = distances <= threshold
    
    # Flood-fill from edges
    grid_mask = np.ones_like(w_grid, dtype=bool)
    visited = np.zeros_like(w_grid, dtype=bool)
    queue = deque()
    
    for i in range(grid_resolution):
        queue.append((i, 0))
        queue.append((i, grid_resolution - 1))
        visited[i, 0] = True
        visited[i, grid_resolution - 1] = True
    
    for j in range(1, grid_resolution - 1):
        queue.append((0, j))
        queue.append((grid_resolution - 1, j))
        visited[0, j] = True
        visited[grid_resolution - 1, j] = True
    
    while queue:
        i, j = queue.popleft()
        
        if not original_data_mask[i, j]:
            grid_mask[i, j] = False
            
            for di, dj in [(-1, 0), (1, 0), (0, -1), (0, 1)]:
                ni, nj = i + di, j + dj
                if (0 <= ni < grid_resolution and 
                    0 <= nj < grid_resolution and 
                    not visited[ni, nj]):
                    visited[ni, nj] = True
                    queue.append((ni, nj))
    
    w_grid[~grid_mask] = np.nan
    
    grid_points = (coords_mean +
                   grid_u[...,None] * tangent1 +
                   grid_v[...,None] * tangent2 +
                   w_grid[...,None] * normal_guess)

    Z, Y, X = component.shape
    sheet_volume = np.zeros_like(component, dtype=bool)

    if use_numba:
        rasterize_surface_numba(grid_points, sheet_volume, samples_per_edge=samples_per_edge)
    else:
        rasterize_surface_dense_sampling_original(grid_points, sheet_volume, samples_per_quad=samples_per_edge)

    sheet_volume = zero_volume_faces(sheet_volume, thickness=5)

    if thickness > 0:
        iterations = max(1, thickness // 2)
        struct_elem = np.array([
            [[0,1,0], [1,1,1], [0,1,0]],
            [[1,1,1], [1,1,1], [1,1,1]],
            [[0,1,0], [1,1,1], [0,1,0]]
        ], dtype=bool)
        sheet_volume = binary_dilation(sheet_volume, structure=struct_elem, iterations=iterations)

    for z in range(Z):
        if np.any(sheet_volume[z]):
            sheet_volume[z] = binary_fill_holes(sheet_volume[z])

    # struct = ndi.generate_binary_structure(3, 3)
    # sheet_volume = ndi.binary_closing(sheet_volume, structure=struct, iterations=1)

    return sheet_volume


def rasterize_surface_dense_sampling_original(grid_points, volume, samples_per_quad=5):
    """Original Python implementation for fallback."""
    grid_resolution = grid_points.shape[0]
    for i in range(grid_resolution - 1):
        for j in range(grid_resolution - 1):
            p1 = grid_points[i, j]
            p2 = grid_points[i+1, j]
            p3 = grid_points[i, j+1]
            p4 = grid_points[i+1, j+1]
            if (np.isnan(p1).any() or np.isnan(p2).any() or
                np.isnan(p3).any() or np.isnan(p4).any()):
                continue
            for u in np.linspace(0, 1, samples_per_quad):
                for v in np.linspace(0, 1, samples_per_quad):
                    point = ((1-u)*(1-v)*p1 + u*(1-v)*p2 + (1-u)*v*p3 + u*v*p4)
                    point = point.round().astype(int)
                    if (0 <= point[0] < volume.shape[0] and
                        0 <= point[1] < volume.shape[1] and
                        0 <= point[2] < volume.shape[2]):
                        volume[point[0], point[1], point[2]] = True


# ============================================================================
# PARALLEL COMPONENT PROCESSING
# ============================================================================

def process_component_wrapper(args):
    """Wrapper for parallel processing of components."""
    component_id, component_mask, grid_resolution, thickness, smoothing, max_distance, samples_per_edge = args
    try:
        fitted = fit_curved_sheet_to_component_optimized(
            component_mask,
            grid_resolution=grid_resolution,
            thickness=thickness,
            smoothing=smoothing,
            max_distance=max_distance,
            use_numba=True,
            adaptive_resolution=True,
            samples_per_edge=samples_per_edge
        )
        return component_id, fitted
    except Exception as e:
        print(f"Error processing component {component_id}: {e}")
        return component_id, component_mask


def _evaluate_component_worker(args):
    """
    Worker for parallel per-component evaluation (erode + quality check + alternatives).
    Returns a dict describing what to write into result_labeled.

    Keys in returned dict:
        'id'              : component id
        'status'          : 'correct' | 'fitted' | 'alternative' | 'removed' | 'lost'
        'main_mask'       : boolean mask to assign label i (or None)
        'extra_components': list of boolean masks for new sub-components (from alternatives)
        'dice'            : float
        'coverage'        : float
    """
    (i, is_correct, component_mask, fitted_after_overlap,
     grid_resolution, thickness, smoothing, max_distance, samples_per_edge,
     alt_min_dice, alt_min_coverage, min_dice, min_coverage,
     alternative_volumes, erosion_iterations, struct_elem) = args

    original_component = component_mask

    # ── Correct components: keep as-is ───────────────────────────────────────
    if is_correct:
        return {
            'id': i, 'status': 'correct',
            'main_mask': fitted_after_overlap,
            'extra_components': [], 'dice': 1.0, 'coverage': 1.0,
        }

    # ── Lost during overlap removal ───────────────────────────────────────────
    if not np.any(fitted_after_overlap):
        print(f"  Component {i}: Lost during overlap removal")
        # Fall through to alternatives below with dice/coverage = 0
        dice, coverage = 0.0, 0.0
        eroded = None
    else:
        # ── Erode ─────────────────────────────────────────────────────────────
        if erosion_iterations > 0:
            eroded = binary_erosion(fitted_after_overlap, structure=struct_elem,
                                    iterations=erosion_iterations)
        else:
            eroded = fitted_after_overlap
        eroded = binary_fill_holes(eroded)

        dice = calculate_dice_score(original_component, eroded)
        coverage = calculate_coverage_score(original_component, eroded)

    if eroded is not None and dice >= min_dice and coverage >= min_coverage:
        print(f"  Component {i}: Dice={dice:.3f}, Coverage={coverage:.3f} ✓ (fitted)")
        return {
            'id': i, 'status': 'fitted',
            'main_mask': eroded,
            'extra_components': [], 'dice': dice, 'coverage': coverage,
        }

    # ── Try alternative volumes ───────────────────────────────────────────────
    if alternative_volumes is not None and len(alternative_volumes) > 0:
        print(f"  Component {i}: Dice={dice:.3f}, Coverage={coverage:.3f} — trying alternatives...")

        all_good_results = []
        remaining_region = original_component.copy()

        for alt_idx, alt_volume in enumerate(alternative_volumes):
            if not np.any(remaining_region):
                break

            print(f"    Alternative {alt_idx+1}/{len(alternative_volumes)} "
                  f"(remaining: {np.sum(remaining_region)} vx)...")

            alt_mask = alt_volume & remaining_region
            if not np.any(alt_mask):
                print(f"      No voxels in alternative within remaining region")
                continue

            alt_labeled = label(alt_mask)
            num_alt_comps = alt_labeled.max()
            print(f"      Found {num_alt_comps} component(s)")

            solved_in_this_alt = np.zeros_like(alt_volume, dtype=bool)
            unsolved_in_this_alt = np.zeros_like(alt_volume, dtype=bool)

            for comp_idx in range(1, num_alt_comps + 1):
                alt_comp = (alt_labeled == comp_idx)

                if np.sum(alt_comp) < 100:
                    print(f"        Component {comp_idx}: Too small ({np.sum(alt_comp)} vx)")
                    continue

                # ── 3-faces check REMOVED ────────────────────────────────────
                # (no face-touching filter applied here)

                try:
                    alt_fitted = fit_curved_sheet_to_component_optimized(
                        alt_comp,
                        grid_resolution=grid_resolution,
                        thickness=thickness,
                        smoothing=smoothing,
                        max_distance=max_distance,
                        use_numba=True,
                        adaptive_resolution=True,
                        samples_per_edge=samples_per_edge
                    )

                    alt_dice = calculate_dice_score(alt_comp, alt_fitted)
                    alt_coverage = calculate_coverage_score(alt_comp, alt_fitted)
                    print(f"        Component {comp_idx}: Dice={alt_dice:.3f}, Cov={alt_coverage:.3f}")

                    if alt_dice >= alt_min_dice and alt_coverage >= alt_min_coverage:
                        all_good_results.append({
                            'fitted': alt_fitted,
                            'dice': alt_dice, 'coverage': alt_coverage,
                            'alt_idx': alt_idx, 'comp_idx': comp_idx,
                            'source_comp': alt_comp,
                        })
                        solved_in_this_alt |= alt_comp
                        print(f"        Component {comp_idx}: ✓ Accepted")
                    else:
                        unsolved_in_this_alt |= alt_comp
                        print(f"        Component {comp_idx}: ✗ Not good enough")

                except Exception as e:
                    print(f"        Component {comp_idx}: Failed to fit ({e})")

            if np.any(solved_in_this_alt):
                remaining_region = unsolved_in_this_alt
                print(f"      Solved {np.sum(solved_in_this_alt)} vx; "
                      f"remaining: {np.sum(unsolved_in_this_alt)} vx")
            else:
                print(f"      No components solved in this alternative")

        if len(all_good_results) > 0:
            print(f"    Total: {len(all_good_results)} good alternative(s)")

            # Overlap detection between alternatives
            if len(all_good_results) > 1:
                alt_fitted_sheets = {idx+1: r['fitted'] for idx, r in enumerate(all_good_results)}
                alt_overlap = detect_overlaps_vectorized(alt_fitted_sheets, len(all_good_results))
                alt_labeled_result = np.zeros_like(alt_volume, dtype=np.int32)
                for idx in range(1, len(all_good_results) + 1):
                    mask = alt_fitted_sheets[idx] & ~alt_overlap
                    alt_labeled_result[mask] = idx

                combined_alternatives = np.zeros_like(alt_volume, dtype=bool)
                for idx in range(1, len(all_good_results) + 1):
                    alt_comp_mask = (alt_labeled_result == idx)
                    if not np.any(alt_comp_mask):
                        continue
                    if erosion_iterations > 0:
                        ea = binary_erosion(alt_comp_mask, structure=struct_elem,
                                            iterations=erosion_iterations)
                    else:
                        ea = alt_comp_mask
                    ea = binary_fill_holes(ea)
                    combined_alternatives |= ea
            else:
                combined_alternatives = all_good_results[0]['fitted']
                if erosion_iterations > 0:
                    combined_alternatives = binary_erosion(
                        combined_alternatives, structure=struct_elem, iterations=erosion_iterations)
                    combined_alternatives = binary_fill_holes(combined_alternatives)

            # No final interpolation pass — beta1 re-interpolation loop will
            # catch any remaining topology issues in a subsequent pass.
            return {
                'id': i, 'status': 'alternative',
                'main_mask': None,
                'extra_components': [combined_alternatives],
                'dice': dice, 'coverage': coverage,
            }

    # ── No valid fit, remove component ───────────────────────────────────────
    print(f"  Component {i}: No valid fit found — REMOVING")
    return {
        'id': i, 'status': 'removed',
        'main_mask': None,
        'extra_components': [], 'dice': dice, 'coverage': coverage,
    }


# ============================================================================
# ITERATIVE BETA1 RE-INTERPOLATION
# ============================================================================

def _reinterpolate_bad_components(
    result_labeled,
    grid_resolution, thickness, smoothing, max_distance, samples_per_edge,
    overlap_buffer, min_dice, min_coverage, alt_min_dice, alt_min_coverage,
    alternative_volumes, use_parallel, n_jobs,
    max_iterations=3,
    debug_output_dir="debug_reinterp",
):
    """
    Check beta1 (= 1 - Euler number) for every component in result_labeled.
    Components with beta1 > 0 are re-processed through the FULL threshold
    pipeline — exactly as in the main loop:
        1. Fit sheet
        2. Overlap removal
        3. Erode + evaluate vs min_dice / min_coverage
        4. If that fails → try alternative volumes vs alt_min_dice / alt_min_coverage

    Repeats up to max_iterations times, stopping early once all β1 ≤ 0.
    Modifies result_labeled in-place and returns it.
    """
    print("\n" + "=" * 70)
    print(f"ITERATIVE BETA1 RE-INTERPOLATION (max {max_iterations} passes)")
    print("=" * 70)

    erosion_iterations = overlap_buffer // 2
    struct_elem = ball(1) if erosion_iterations > 0 else None

    for iteration in range(max_iterations):
        print(f"\n--- Pass {iteration + 1}/{max_iterations} ---")

        # ── Identify bad components ───────────────────────────────────────────
        current_binary = result_labeled > 0
        check_labeled = label(current_binary)
        num_check = check_labeled.max()

        bad_ids = []
        for cid in range(1, num_check + 1):
            comp = (check_labeled == cid)
            chi = euler_number(comp.astype(int), connectivity=1)
            beta1 = 1 - chi
            if beta1 > 0:
                bad_ids.append(cid)

        if not bad_ids:
            print(f"  All {num_check} components have β1≤0 — done early!")
            break

        print(f"  {len(bad_ids)}/{num_check} components have β1>0")

        # ── Step A: Fit sheets in parallel ────────────────────────────────────
        fit_args = [
            (cid,
             (check_labeled == cid),
             grid_resolution,
             thickness + overlap_buffer,
             smoothing,
             max_distance,
             samples_per_edge)
            for cid in bad_ids
        ]

        if use_parallel and len(fit_args) > 1:
            max_workers = n_jobs if n_jobs > 0 else None
            with ThreadPoolExecutor(max_workers=max_workers) as executor:
                fit_results = list(executor.map(process_component_wrapper, fit_args))
        else:
            fit_results = [process_component_wrapper(a) for a in fit_args]

        fitted_sheets = {cid: fitted for cid, fitted in fit_results}

        # ── Step B: Overlap detection among re-fitted sheets ──────────────────
        if len(fitted_sheets) > 1:
            # Build a 1-indexed dict for detect_overlaps_vectorized
            id_to_idx = {cid: idx + 1 for idx, cid in enumerate(bad_ids)}
            idx_sheets = {id_to_idx[cid]: fitted_sheets[cid] for cid in bad_ids}
            overlap_mask = detect_overlaps_vectorized(idx_sheets, len(bad_ids))
        else:
            shape = list(fitted_sheets.values())[0].shape
            overlap_mask = np.zeros(shape, dtype=bool)

        fitted_after_overlap = {
            cid: fitted_sheets[cid] & ~overlap_mask for cid in bad_ids
        }

        # ── Step C: Evaluate each bad component through full threshold pipeline
        eval_args = [
            (cid,
             False,                          # is_correct = False
             (check_labeled == cid),         # component_mask (original)
             fitted_after_overlap[cid],      # fitted after overlap removal
             grid_resolution, thickness + overlap_buffer, smoothing, max_distance, samples_per_edge,
             alt_min_dice, alt_min_coverage, min_dice, min_coverage,
             alternative_volumes, erosion_iterations, struct_elem)
            for cid in bad_ids
        ]

        if use_parallel and len(eval_args) > 1:
            max_workers = n_jobs if n_jobs > 0 else None
            with ThreadPoolExecutor(max_workers=max_workers) as executor:
                eval_results = list(executor.map(_evaluate_component_worker, eval_args))
        else:
            eval_results = [_evaluate_component_worker(a) for a in eval_args]

        # ── Step D: Update result_labeled ─────────────────────────────────────
        next_label = result_labeled.max() + 1

        for res in eval_results:
            cid = res['id']
            old_mask = (check_labeled == cid)

            # Find the dominant original label this geometric region carried
            orig_labels = result_labeled[old_mask]
            dominant_label = (
                int(np.bincount(orig_labels[orig_labels > 0]).argmax())
                if np.any(orig_labels > 0) else 0
            )
            if dominant_label == 0:
                dominant_label = next_label
                next_label += 1

            # Clear old voxels for this component
            result_labeled[old_mask] = 0

            if res['status'] in ('fitted', 'correct') and res['main_mask'] is not None:
                result_labeled[res['main_mask'] & (result_labeled == 0)] = dominant_label

            elif res['status'] == 'alternative':
                for extra_mask in res['extra_components']:
                    result_labeled[extra_mask & (result_labeled == 0)] = next_label
                    next_label += 1

            else:  # 'removed' — voxels already cleared, component is gone
                print(f"    Component {cid}: removed after re-interpolation")

        # ── Quick per-pass summary ─────────────────────────────────────────────
        statuses = [r['status'] for r in eval_results]
        print(f"  Pass {iteration + 1} results: "
              f"fitted={statuses.count('fitted')}, "
              f"alternative={statuses.count('alternative')}, "
              f"removed={statuses.count('removed')}")

    return result_labeled


# ============================================================================
# MAIN PARALLEL PROCESSING FUNCTION
# ============================================================================

def process_multiple_components_parallel(
    volume,
    alternative_volumes=None,
    grid_resolution=80,
    thickness=3,
    smoothing=2.0,
    overlap_buffer=2,
    min_coverage=0.60,
    min_dice=0.6,
    alt_min_coverage=None,
    alt_min_dice=None,
    max_distance=10,
    use_parallel=True,
    n_jobs=-1,
    samples_per_edge=8,
    max_reinterp_iterations=1,
    debug_output_dir="debug_reinterp",
):
    """
    Optimized parallel processing with:
      • Euler-based pre-filtering (skip topologically correct components)
      • Numba JIT rasterization
      • Parallel fitting AND parallel per-component evaluation (incl. alternatives)
      • Iterative beta1 re-interpolation loop (up to max_reinterp_iterations)
      • 3-faces check REMOVED
    """
    labeled_volume = label(volume)
    num_components = labeled_volume.max()

    if alt_min_coverage is None:
        alt_min_coverage = min_coverage
    if alt_min_dice is None:
        alt_min_dice = min_dice

    print(f"Processing {num_components} components...")
    print(f"Optimizations: Numba=True, Parallel={use_parallel}, AdaptiveRes=True")
    if alternative_volumes is not None:
        print(f"Using {len(alternative_volumes)} alternative volumes for fallback")
        print(f"Main thresholds:        Dice≥{min_dice:.2f}, Coverage≥{min_coverage:.2f}")
        print(f"Alternative thresholds: Dice≥{alt_min_dice:.2f}, Coverage≥{alt_min_coverage:.2f}")

    # ── Extract component masks ───────────────────────────────────────────────
    component_masks = {i: (labeled_volume == i) for i in range(1, num_components + 1)}

    # ========================================================================
    # STEP 1: Euler-based topology analysis
    # ========================================================================
    print("\n" + "=" * 70)
    print("STEP 1: Euler-based topology analysis")
    print("=" * 70)

    correct_components = []
    needs_interpolation = []

    for i in range(1, num_components + 1):
        chi = euler_number(component_masks[i].astype(int), connectivity=1)
        beta1 = 1 - chi
        if beta1 <= 0:
            correct_components.append(i)
            print(f"  Component {i}: β1={beta1} (χ={chi}) ✓ CORRECT — keeping as-is")
        else:
            needs_interpolation.append(i)
            print(f"  Component {i}: β1={beta1} (χ={chi}) ⚠  NEEDS INTERPOLATION")

    print(f"\nSummary: {len(correct_components)} correct, "
          f"{len(needs_interpolation)} need interpolation")

    # ========================================================================
    # STEP 2: Fit sheets to components that need it (parallel)
    # ========================================================================
    fitted_sheets = {}

    # Correct components use original masks unchanged
    for i in correct_components:
        fitted_sheets[i] = component_masks[i]

    if len(needs_interpolation) > 0:
        print("\n" + "=" * 70)
        print(f"STEP 2: Fitting sheets for {len(needs_interpolation)} component(s)")
        print("=" * 70)

        fit_args = [
            (i, component_masks[i], grid_resolution,
             thickness + overlap_buffer, smoothing, max_distance, samples_per_edge)
            for i in needs_interpolation
        ]

        if use_parallel and len(needs_interpolation) > 1:
            print("  Running in parallel...")
            max_workers = n_jobs if n_jobs > 0 else None
            with ThreadPoolExecutor(max_workers=max_workers) as executor:
                fit_results = list(executor.map(process_component_wrapper, fit_args))
        else:
            fit_results = [process_component_wrapper(a) for a in fit_args]

        for cid, fitted in fit_results:
            fitted_sheets[cid] = fitted

    # ========================================================================
    # STEP 3: Overlap detection (vectorized, across ALL components)
    # ========================================================================
    print("\n" + "=" * 70)
    print("STEP 3: Detecting overlaps (vectorized)")
    print("=" * 70)

    overlap_mask = detect_overlaps_vectorized(fitted_sheets, num_components)
    print(f"Removed {np.sum(overlap_mask)} overlapping voxels")

    # Build per-component overlap-free slices for evaluation
    fitted_after_overlap = {}
    for i in range(1, num_components + 1):
        fitted_after_overlap[i] = fitted_sheets[i] & ~overlap_mask

    # ========================================================================
    # STEP 4: Evaluate components — erode, check quality, try alternatives
    #         Run ALL components in parallel (correct ones short-circuit instantly)
    # ========================================================================
    print("\n" + "=" * 70)
    print("STEP 4: Evaluating & rescuing components (parallel)")
    print("=" * 70)

    erosion_iterations = overlap_buffer // 2
    struct_elem = ball(1) if erosion_iterations > 0 else None

    eval_args = [
        (i,
         i in correct_components,
         component_masks[i],
         fitted_after_overlap[i],
         grid_resolution, thickness + overlap_buffer, smoothing, max_distance, samples_per_edge,
         alt_min_dice, alt_min_coverage, min_dice, min_coverage,
         alternative_volumes, erosion_iterations, struct_elem)
        for i in range(1, num_components + 1)
    ]

    if use_parallel and num_components > 1:
        print(f"  Running evaluation for {num_components} components in parallel...")
        max_workers = n_jobs if n_jobs > 0 else None
        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            eval_results = list(executor.map(_evaluate_component_worker, eval_args))
    else:
        eval_results = [_evaluate_component_worker(a) for a in eval_args]

    # ========================================================================
    # STEP 5: Assemble result_labeled from evaluation results
    # ========================================================================
    print("\n" + "=" * 70)
    print("STEP 5: Assembling final result")
    print("=" * 70)

    result_labeled = np.zeros_like(volume, dtype=np.int32)
    dice_scores = {}
    coverage_scores = {}

    next_label = num_components + 1
    kept_correct = 0
    kept_fitted = 0
    kept_alternative = 0
    removed = 0

    for res in eval_results:
        i = res['id']
        dice_scores[i] = res['dice']
        coverage_scores[i] = res['coverage']

        if res['status'] == 'correct':
            if res['main_mask'] is not None:
                result_labeled[res['main_mask']] = i
            kept_correct += 1

        elif res['status'] == 'fitted':
            if res['main_mask'] is not None:
                result_labeled[res['main_mask'] & (result_labeled == 0)] = i
            kept_fitted += 1

        elif res['status'] == 'alternative':
            for extra_mask in res['extra_components']:
                result_labeled[extra_mask & (result_labeled == 0)] = next_label
                next_label += 1
            kept_alternative += 1

        else:  # 'removed' or 'lost'
            removed += 1
            print(f"  Component {i}: {res['status'].upper()}")

    print(f"\n  Correct (β1=0):   {kept_correct}")
    print(f"  Fitted:           {kept_fitted}")
    print(f"  Via alternatives: {kept_alternative}")
    print(f"  Removed:          {removed}")
    print(f"  Total kept:       {kept_correct + kept_fitted + kept_alternative}/{num_components}")

    # ========================================================================
    # STEP 6: Iterative beta1 re-interpolation (parallel, up to N passes)
    # ========================================================================
    result_labeled = _reinterpolate_bad_components(
        result_labeled,
        grid_resolution=grid_resolution,
        thickness=thickness,
        smoothing=smoothing,
        max_distance=max_distance,
        samples_per_edge=samples_per_edge,
        overlap_buffer=overlap_buffer,
        min_dice=min_dice,
        min_coverage=min_coverage,
        alt_min_dice=alt_min_dice,
        alt_min_coverage=alt_min_coverage,
        alternative_volumes=alternative_volumes,
        use_parallel=use_parallel,
        n_jobs=n_jobs,
        max_iterations=max_reinterp_iterations,
        debug_output_dir=debug_output_dir,
    )

    # ========================================================================
    # Final summary
    # ========================================================================
    result_binary = result_labeled > 0
    final_labeled = label(result_binary)
    final_num = final_labeled.max()

    valid_dice = [v for v in dice_scores.values() if isinstance(v, (int, float))]
    avg_dice = float(np.mean(valid_dice)) if valid_dice else 0.0

    print(f"\n{'=' * 70}")
    print(f"FINAL SUMMARY")
    print(f"{'=' * 70}")
    print(f"Final component count : {final_num}")
    print(f"Average Dice score    : {avg_dice:.3f}")

    return result_binary, result_labeled, dice_scores, coverage_scores


# ============================================================================
# HELPER FUNCTIONS
# ============================================================================

def calculate_dice_score(mask1, mask2):
    """Calculate Dice coefficient between two binary masks."""
    intersection = np.sum(mask1 & mask2)
    sum_masks = np.sum(mask1) + np.sum(mask2)
    if sum_masks == 0:
        return 0.0
    return 2.0 * intersection / sum_masks


def calculate_coverage_score(original, fitted):
    """Calculate how well the fitted sheet covers the original positive pixels."""
    original_pixels = np.sum(original)
    if original_pixels == 0:
        return 0.0
    return np.sum(original & fitted) / original_pixels

In [ ]:
# ============================================================
# OPTIMIZED DUAL-GPU ENSEMBLE INFERENCE
# Model 1 on GPU:0, Model 2 on GPU:1, Parallel Processing
# ============================================================

# ============================================================
# 1. INFERENCE CONFIGURATION
# ============================================================

CONFIG = {
    # --- MODEL 1 (GPU:0) ---
    "model1_folder": "/kaggle/input/models/nguyncdngs/nnunet-resenc/pytorch/default/3/nnUNet_ResEnc",
    "model1_checkpoint": "checkpoint_final.pth",
    "model1_weight": 0.6,
    "model1_device": "cuda:0",  # First T4
    
    # --- MODEL 2 (GPU:1) ---
    "model2_folder": "/kaggle/input/models/nguyncdngs/nnunet-resenc/pytorch/default/2/nnUNet_ResEnc",
    "model2_checkpoint": "checkpoint_final.pth",
    "model2_weight": 0.4,
    "model2_device": "cuda:1",  # Second T4
    
    # Đường dẫn data
    "input_dir": "/kaggle/input/vesuvius-challenge-surface-detection/test_images",
    "output_zip": "submission.zip",
    
    # Cấu hình Predictor
    "use_folds": ("all",),          
    "tile_step_size": 0.5,      
    "use_gaussian": True,        
    "use_mirroring": True,
    
    # Spacing
    "spacing": [1.0, 1.0, 1.0],

    # --- POST-PROCESSING CONFIG ---
    "enable_postproc": True,
    "T_low": 0.2,
    "T_high": 0.83,
    "z_radius": 1,
    "xy_radius": 0,
    "min_object_size": 2000,
}

# ============================================================
# 2. IMPORTS
# ============================================================

import os
import glob
import zipfile
import numpy as np
import torch
import tifffile
from tqdm import tqdm
from skimage import morphology
import scipy.ndimage as ndi
from concurrent.futures import ThreadPoolExecutor
import threading

from nnunetv2.inference.predict_from_raw_data import nnUNetPredictor
from nnunetv2.imageio.tif_reader_writer import Tiff3DIO

# ============================================================
# 3. POST-PROCESSING FUNCTIONS
# ============================================================

def build_anisotropic_struct(z_radius: int, xy_radius: int):
    """Tạo kernel 3D hình elip/trụ để đóng lỗ hổng."""
    z, r = z_radius, xy_radius

    if z == 0 and r == 0:
        return None

    if z == 0 and r > 0:
        size = 2 * r + 1
        struct = np.zeros((1, size, size), dtype=bool)
        cy, cx = r, r
        for dy in range(-r, r + 1):
            for dx in range(-r, r + 1):
                if dy * dy + dx * dx <= r * r:
                    struct[0, cy + dy, cx + dx] = True
        return struct

    if z > 0 and r == 0:
        struct = np.zeros((2 * z + 1, 1, 1), dtype=bool)
        struct[:, 0, 0] = True
        return struct

    depth = 2 * z + 1
    size = 2 * r + 1
    struct = np.zeros((depth, size, size), dtype=bool)
    cz, cy, cx = z, r, r
    for dz in range(-z, z + 1):
        for dy in range(-r, r + 1):
            for dx in range(-r, r + 1):
                if dy * dy + dx * dx <= r * r:
                    struct[cz + dz, cy + dy, cx + dx] = True
    return struct

def topo_postprocess(
                    probs,          # (D, H, W)
                    T_low=0.6,
                    T_high=0.9,
                    z_radius=1,
                    xy_radius=1,
                    dust_min_size=500,
                ):

     # Step 1: 3D Hysteresis
    strong = probs >= T_high
    weak   = probs >= T_low

    if not strong.any():
        return np.zeros_like(probs, dtype=np.uint8)

    struct_hyst = ndi.generate_binary_structure(3, 3)
    mask = ndi.binary_propagation(strong, mask=weak, structure=struct_hyst)

    if not mask.any():
        return np.zeros_like(probs, dtype=np.uint8)

    # Step 2: 3D Anisotropic Closing
    if z_radius > 0 or xy_radius > 0:
        struct_close = build_anisotropic_struct(z_radius, xy_radius)
        if struct_close is not None:
            mask = ndi.binary_closing(mask, structure=struct_close)

    # Step 3: Dust Removal
    if dust_min_size > 0:
        mask = morphology.remove_small_objects(mask.astype(bool), min_size=dust_min_size)

    return mask.astype(np.uint8)

# ============================================================
# 4. PARALLEL PREDICTION CLASS
# ============================================================

class ParallelPredictor:
    """Wrapper to run predictions in parallel on separate GPUs."""
    
    def __init__(self, predictor, device_name):
        self.predictor = predictor
        self.device_name = device_name
        self.lock = threading.Lock()
    
    def predict(self, image, properties):
        """Thread-safe prediction on assigned GPU."""
        with self.lock:
            ret = self.predictor.predict_single_npy_array(
                image, 
                properties, 
                segmentation_previous_stage=None, 
                output_file_truncated=None, 
                save_or_return_probabilities=True
            )
        return ret

# ============================================================
# 5. MAIN INFERENCE ENGINE
# ============================================================

def run_inference():
    print("--> [INIT] Setting up Dual-GPU Parallel Ensemble Inference...")
    
    # 1. Check GPU availability
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA not available!")
    
    gpu_count = torch.cuda.device_count()
    print(f"--> [GPU CHECK] Found {gpu_count} GPU(s)")
    
    if gpu_count < 2:
        print(f"--> [WARNING] Only {gpu_count} GPU found. Falling back to single GPU mode.")
        device1 = torch.device('cuda:0')
        device2 = torch.device('cuda:0')
    else:
        device1 = torch.device(CONFIG["model1_device"])
        device2 = torch.device(CONFIG["model2_device"])
        print(f"--> [GPU ASSIGNMENT] Model 1 -> {device1}, Model 2 -> {device2}")

    # 2. Initialize Model 1 on GPU:0
    print(f"--> [MODEL 1] Loading on {device1}...")
    predictor1 = nnUNetPredictor(
        tile_step_size=CONFIG["tile_step_size"],
        use_gaussian=CONFIG["use_gaussian"],
        use_mirroring=CONFIG["use_mirroring"],
        perform_everything_on_device=True,
        device=device1,
        verbose=False,
        verbose_preprocessing=False,
        allow_tqdm=False
    )
    
    predictor1.initialize_from_trained_model_folder(
        CONFIG["model1_folder"],
        use_folds=CONFIG["use_folds"],
        checkpoint_name=CONFIG["model1_checkpoint"]
    )
    print(f"   -> Model 1 loaded on {device1}")

    # 3. Initialize Model 2 on GPU:1
    print(f"--> [MODEL 2] Loading on {device2}...")
    predictor2 = nnUNetPredictor(
        tile_step_size=CONFIG["tile_step_size"],
        use_gaussian=CONFIG["use_gaussian"],
        use_mirroring=CONFIG["use_mirroring"],
        perform_everything_on_device=True,
        device=device2,
        verbose=False,
        verbose_preprocessing=False,
        allow_tqdm=False
    )
    
    predictor2.initialize_from_trained_model_folder(
        CONFIG["model2_folder"],
        use_folds=CONFIG["use_folds"],
        checkpoint_name=CONFIG["model2_checkpoint"]
    )
    print(f"   -> Model 2 loaded on {device2}")
    
    # 4. Wrap predictors for parallel execution
    parallel_pred1 = ParallelPredictor(predictor1, device1)
    parallel_pred2 = ParallelPredictor(predictor2, device2)
    
    # 5. Setup Image Reader
    reader = Tiff3DIO()
    
    # 6. Prepare Input Files
    test_files = sorted(glob.glob(os.path.join(CONFIG["input_dir"], "*.tif")))
    if not test_files:
        print("--> [WARNING] No .tif files found.")
        return 
    
    print(f"--> [DATA] Found {len(test_files)} files to process.")
    print(f"--> [ENSEMBLE] Weights: Model1={CONFIG['model1_weight']}, Model2={CONFIG['model2_weight']}")

    # 7. Processing Loop with Parallel Execution
    with zipfile.ZipFile(CONFIG["output_zip"], 'w', zipfile.ZIP_DEFLATED) as zf:
        
        for file_path in tqdm(test_files, desc="Parallel Dual-GPU Inference"):
            filename = os.path.basename(file_path)
            
            # Read image
            image, _ = reader.read_images([file_path])
            
            properties = {
                'spacing': CONFIG['spacing']
            }

            # --- PARALLEL PREDICTION ON BOTH GPUS ---
            with ThreadPoolExecutor(max_workers=2) as executor:
                # Submit both predictions simultaneously
                future1 = executor.submit(parallel_pred1.predict, image, properties)
                future2 = executor.submit(parallel_pred2.predict, image, properties)
                
                # Wait for both to complete
                ret1 = future1.result()
                ret2 = future2.result()

            # --- Extract Probabilities ---
            if isinstance(ret1, tuple):
                _, probabilities1 = ret1
            else:
                probabilities1 = None

            if isinstance(ret2, tuple):
                _, probabilities2 = ret2
            else:
                probabilities2 = None

            # --- ENSEMBLE PROBABILITIES ---
            if probabilities1 is not None and probabilities2 is not None:
                # Weighted ensemble
                total_weight = CONFIG["model1_weight"] + CONFIG["model2_weight"]
                w1 = CONFIG["model1_weight"] / total_weight
                w2 = CONFIG["model2_weight"] / total_weight
                
                ensembled_probs = w1 * probabilities1 + w2 * probabilities2
                prob_map = ensembled_probs[1]
                # Apply multiple threshold levels
                pred = topo_postprocess(prob_map, T_low=0.2,
                                            T_high=0.83,
                                            z_radius=1,
                                            xy_radius=0,
                                            dust_min_size=100)
                pred = zero_volume_faces(pred.astype(bool), thickness=3)
                pred = morphology.remove_small_objects(
                    pred.astype(bool),
                    min_size=1000,
                    connectivity=3
                )
                
                pred3 = topo_postprocess(prob_map, T_low=0.5,
                                            T_high=0.83,
                                            z_radius=1,
                                            xy_radius=0,
                                            dust_min_size=100)
                pred3 = zero_volume_faces(pred3.astype(bool), thickness=3)
                pred3 = morphology.remove_small_objects(
                    pred3.astype(bool),
                    min_size=1000,
                    connectivity=3
                )
                
                pred5 = topo_postprocess(prob_map, T_low=0.7,
                                            T_high=0.83,
                                            z_radius=1,
                                            xy_radius=0,
                                            dust_min_size=100)
                pred5 = zero_volume_faces(pred5.astype(bool), thickness=3)
                pred5 = morphology.remove_small_objects(
                    pred5.astype(bool),
                    min_size=1000,
                    connectivity=3
                )
                
                # Fill holes
                pred = binary_fill_holes(pred.astype(bool))
                
                # Crop borders
                pred = pred[3:-3, 3:-3, 3:-3]
                pred3 = pred3[3:-3, 3:-3, 3:-3]
                pred5 = pred5[3:-3, 3:-3, 3:-3]
                
                # Advanced component processing
                pred = process_multiple_components_parallel(
                    pred, 
                    grid_resolution=100, 
                    thickness=3, 
                    smoothing=1.0,
                    overlap_buffer=0, 
                    min_coverage=0.65, 
                    min_dice=0.7,
                    max_distance=10, 
                    alternative_volumes=[pred3, pred5], 
                    n_jobs=-1,
                    alt_min_coverage=0.75,
                    alt_min_dice=0.45,
                    samples_per_edge=8
                )
                
                # Final cleanup
                pred = morphology.remove_small_objects(
                    pred[0], 
                    min_size=2000, 
                    connectivity=3
                )
                
                # Restore border padding
                border = 3
                pred = np.pad(
                    pred,
                    pad_width=(
                        (border, border),
                        (border, border),
                        (border, border)
                    ),
                    mode="constant",
                    constant_values=0
                )
                
                pred_uint8 = pred.astype(np.uint8)
                
            else:
                # Fallback without post-processing
                if segmentation_mask.ndim == 4:
                    segmentation_mask = segmentation_mask[0]
                pred_uint8 = segmentation_mask.astype(np.uint8)

            # --- Save to Zip ---
            temp_output_path = filename 
            tifffile.imwrite(temp_output_path, pred_uint8)            
            zf.write(temp_output_path, arcname=filename)
            
            if os.path.exists(temp_output_path):
                os.remove(temp_output_path)

    print(f"\n--> [DONE] Parallel Dual-GPU Inference Complete!")
    print(f"    Output: {CONFIG['output_zip']}")

if __name__ == "__main__":
    run_inference()